In [29]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
import os
import logging
from tqdm import tqdm

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("weather_data_collection.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


def merge_monthly_data(output_dir="weather_data", final_dir="all_stations"):
    """
    Merge monthly data into single files for each station
    
    Parameters:
    - output_dir: Directory containing monthly data
    - final_dir: Directory to save merged files
    """
    # Create output directory
    create_directory(final_dir)
    
    # Dictionary to store data for each station
    station_files = {}
    
    # List all month directories
    month_dirs = [d for d in os.listdir(output_dir) if os.path.isdir(os.path.join(output_dir, d)) and d.startswith("20")]
    
    # Find all station files
    for month_dir in month_dirs:
        month_path = os.path.join(output_dir, month_dir)
        
        for filename in os.listdir(month_path):
            if not filename.endswith(".csv") or filename == "station_metadata.csv":
                continue
            
            # Extract station ID from filename
            station_id = filename.split("_")[0]
            
            if station_id not in station_files:
                station_files[station_id] = []
            
            filepath = os.path.join(month_path, filename)
            station_files[station_id].append(filepath)
    
    # Merge data for each station
    for station_id, file_paths in tqdm(station_files.items(), desc="Merging station data"):
        # Read and concatenate all files
        dfs = []
        station_name = None
        
        for filepath in file_paths:
            try:
                df = pd.read_csv(filepath)
                dfs.append(df)
                
                # Extract station name from filename if not already set
                if station_name is None:
                    filename = os.path.basename(filepath)
                    parts = filename.split("_")
                    station_name = "_".join(parts[1:]).replace(".csv", "")
            except Exception as e:
                logger.error(f"Error reading {filepath}: {e}")
        
        if not dfs:
            logger.warning(f"No valid data files for station {station_id}")
            continue
        
        # Concatenate all dataframes
        merged_df = pd.concat(dfs, ignore_index=True)
        
        # Convert timestamp to datetime for sorting
        merged_df['timestamp'] = pd.to_datetime(merged_df['timestamp'])
        
        # Sort by timestamp
        merged_df = merged_df.sort_values('timestamp')
        
        # Remove duplicates
        merged_df = merged_df.drop_duplicates(subset=['timestamp'])
        
        # Create output filename
        filename = f"{station_id}_{station_name}.csv"
        filepath = os.path.join(final_dir, filename)
        
        # Save to CSV
        merged_df.to_csv(filepath, index=False)
        logger.info(f"Saved merged data with {len(merged_df)} records to {filepath}")

In [30]:
def create_directory(directory):
    """Create directory if it doesn't exist"""
    if not os.path.exists(directory):
        os.makedirs(directory)
        logger.info(f"Created directory: {directory}")

In [31]:
import os
import pandas as pd
from pathlib import Path

def combine_station_metadata(output_dir):
    """
    Combines all station_metadata.csv files from month directories and removes duplicates.
    
    Args:
        output_dir (str): The parent directory containing month directories
    
    Returns:
        pd.DataFrame: Combined dataframe with unique station metadata
    """
    # List all month directories
    month_dirs = [d for d in os.listdir(output_dir) if os.path.isdir(os.path.join(output_dir, d)) and d.startswith("20")]
    
    # Initialize an empty list to store dataframes
    all_metadata_dfs = []
    
    # Process each month directory
    for month_dir in month_dirs:
        month_path = os.path.join(output_dir, month_dir)
        metadata_file = os.path.join(month_path, "station_metadata.csv")
        
        # Check if the metadata file exists
        if os.path.exists(metadata_file):
            try:
                # Read the CSV file
                df = pd.read_csv(metadata_file)
                all_metadata_dfs.append(df)
                print(f"Loaded metadata from {month_dir} - {len(df)} stations")
            except Exception as e:
                print(f"Error reading {metadata_file}: {e}")
    
    # Combine all dataframes if we found any
    if all_metadata_dfs:
        combined_df = pd.concat(all_metadata_dfs, ignore_index=True)
        
        # Remove duplicates
        original_count = len(combined_df)
        combined_df = combined_df.drop_duplicates(subset=['id'], keep='first')
        deduped_count = len(combined_df)
        
        print(f"\nTotal stations before deduplication: {original_count}")
        print(f"Total unique stations: {deduped_count}")
        print(f"Removed {original_count - deduped_count} duplicates")
        
        # Save the combined and deduplicated data
        output_file = os.path.join(output_dir, "combined_station_metadata.csv")
        combined_df.to_csv(output_file, index=False)
        print(f"\nCombined metadata saved to: {output_file}")
        
        return combined_df
    else:
        print("No station_metadata.csv files found in month directories.")
        return None

if __name__ == "__main__":
    # You can modify this path to your actual output directory
    output_directory = input("Enter the path to the directory containing month folders: ")
    combine_station_metadata(output_directory)

Loaded metadata from 2024_01 - 64 stations
Loaded metadata from 2024_02 - 63 stations
Loaded metadata from 2024_03 - 63 stations
Loaded metadata from 2024_04 - 63 stations
Loaded metadata from 2024_05 - 64 stations
Loaded metadata from 2024_06 - 64 stations
Loaded metadata from 2024_07 - 64 stations
Loaded metadata from 2024_08 - 63 stations
Loaded metadata from 2024_09 - 62 stations
Loaded metadata from 2024_10 - 62 stations
Loaded metadata from 2024_11 - 64 stations
Loaded metadata from 2024_11_1 - 61 stations
Loaded metadata from 2024_12 - 64 stations
Loaded metadata from 2025_01 - 65 stations
Loaded metadata from 2025_02 - 63 stations
Loaded metadata from 2025_03 - 63 stations
Loaded metadata from 2025_04 - 62 stations

Total stations before deduplication: 1074
Total unique stations: 70
Removed 1004 duplicates

Combined metadata saved to: singapore_weather_monthly\combined_station_metadata.csv


In [32]:
merge_monthly_data("singapore_weather_monthly", "singapore_weather_combined")

2025-04-29 14:36:46,564 - INFO - Created directory: singapore_weather_combined
Merging station data:  11%|█▏        | 8/70 [00:01<00:08,  7.16it/s]2025-04-29 14:36:47,856 - INFO - Saved merged data with 7737 records to singapore_weather_combined\S114_Choa_Chu_Kang_Avenue_4.csv
2025-04-29 14:36:47,992 - INFO - Saved merged data with 11266 records to singapore_weather_combined\S115_Tuas_South_Avenue_3.csv
Merging station data:  17%|█▋        | 12/70 [00:01<00:07,  8.18it/s]2025-04-29 14:36:48,255 - INFO - Saved merged data with 202 records to singapore_weather_combined\S118_Handy_Road.csv
2025-04-29 14:36:48,357 - INFO - Saved merged data with 11361 records to singapore_weather_combined\S119_Nicoll_Highway.csv
Merging station data:  20%|██        | 14/70 [00:01<00:05, 10.74it/s]2025-04-29 14:36:48,379 - INFO - Saved merged data with 2617 records to singapore_weather_combined\S120_Holland_Road.csv
2025-04-29 14:36:48,507 - INFO - Saved merged data with 10605 records to singapore_weather_c